# validation check — delta buckets (market − lrr) → blanket HOME, early season

Split matches into `N_BUCKETS` (base 6) equal-count buckets of `delta = market − elo(lrr)`; boundaries
fit on the **train fold only**. Rule: bet the last bucket (market strongly above elo) on HOME. Blanket model. Population = early season (gw ≤ 8).
Rough-draft partition across three notebooks: first bucket → AWAY, middle → DRAW, last bucket → HOME.
`N_BUCKETS` is a knob — swept 3..14 at the bottom. Region/outcome fixed from the full-sample look →
meta-DoF; gates test time/league robustness.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE — model space (ignored by the blanket model; kept for dropna) -----
MARKET = "mrkt_home_away_impl_diff"
ELO    = "teams_elo_diff_lrr_impl_diff"
FEATURES = [MARKET, ELO]

In [3]:
# --- TARGET ------------------------------------------------------------------
OUTCOME = "t_home_flg"
IMPLIED = "mrkt_home_impl"

In [4]:
# --- ABT FILTERING — early season (gw <= 8) ---------------------------------
MAX_GAMEWEEK = 8
matches = abt[abt["gameweek"] <= MAX_GAMEWEEK].dropna(
    subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()
print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 1544   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER — delta buckets (bet last bucket) ----------------------------
# delta = market - elo(lrr). N_BUCKETS equal-count buckets, boundaries fit on TRAIN only. N_BUCKETS is
# a knob (swept below). Contract: segmenter(train) -> labeler ; labeler(rows) -> Series (NaN = skip).
N_BUCKETS = 6

def _delta(df):
    return df[MARKET] - df[ELO]

def region_segmenter(n_buckets):
    def segmenter(train):
        hi = _delta(train).quantile((n_buckets - 1) / n_buckets)
        def labeler(rows):
            dd = _delta(rows)
            lab = pd.Series(np.nan, index=rows.index, dtype="float64")
            lab[dd > hi] = 1.0
            return lab
        return labeler
    return segmenter

segmenter = region_segmenter(N_BUCKETS)

# Self-description (in-sample): the N_BUCKETS delta buckets; bet buckets marked.
_d = _delta(matches)
_b = pd.qcut(_d, N_BUCKETS, labels=False)
print(f"in-sample delta buckets (N_BUCKETS={N_BUCKETS}; bet last bucket -> HOME)")
print(" bucket      n   delta    win%  impl%   edge     ROI   bet")
for b in range(N_BUCKETS):
    m = matches[_b == b]
    wr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    bet = "HOME" if b == N_BUCKETS - 1 else "  --"
    print(f" {b:>6} {len(m):>7}  {_d[_b == b].mean():+6.3f}   {wr*100:5.1f}  {im*100:5.1f}  "
          f"{(wr-im)*100:+5.1f}  {pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {bet}")

in-sample delta buckets (N_BUCKETS=6; bet last bucket -> HOME)
 bucket      n   delta    win%  impl%   edge     ROI   bet
      0     258  -0.159    29.1   34.7   -5.6  -0.162    --
      1     258  -0.066    37.2   40.3   -3.1  -0.077    --
      2     258  -0.014    45.3   45.7   -0.3  -0.007    --
      3     256  +0.030    42.2   48.4   -6.2  -0.128    --
      4     257  +0.076    49.8   53.5   -3.7  -0.070    --
      5     257  +0.164    56.4   52.2   +4.2  +0.080  HOME


In [6]:
# --- MODEL — blanket bettor -------------------------------------------------
class BlanketBettor:
    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        return self
    def predict_proba(self, X):
        n = len(X)
        return np.column_stack([np.zeros(n), np.ones(n)])

def make_model():
    return BlanketBettor()

BUFFER = 0.0

## Summary

In [7]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="elodelta_home_last_blanket", space="elo_delta", target="home",
              abt_filter="major, gw<=8, dropna",
              segmentation=f"delta=market-lrr, {N_BUCKETS} qcut buckets (train-fit), bet last bucket",
              segment_filter=f"last 1 of {N_BUCKETS} (blanket)")

elodelta_home_last_blanket   pooled +0.082 (254 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           PASS  (+3/4 seasons)
    2223 -0.109    2324 +0.129    2425 +0.133    2526 +0.169

GATE 2  league — ROI by league / drop best   PASS
    france +0.259    spain +0.147    italy +0.060    england +0.002    germany -0.021
    pooled without best league (france): +0.031

GATE 3  walk-forward — ROI by test season    PASS
    2425 +0.133    2526 +0.169


## Extra checks — below the template

In [8]:
# EXTRA CHECK — knob robustness: number of delta buckets n = 3..14 (bet last bucket -> HOME).
# A real edge survives a range of n; one that appears at a single split is fragile.
print(" n_buckets   pooled    profit    bets   seasons+   bet_frac")
for n in range(3, 15):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=region_segmenter(n), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {n:<9} {pooled_roi(b['implied'], b['won']):+.4f}  {(b['won'] - b['implied']).sum():+8.2f}  "
          f"{len(b):>5}   +{int((ps > 0).sum())}/{len(ps)}      {1 / n:.2f}")

 n_buckets   pooled    profit    bets   seasons+   bet_frac
 3         +0.0013     +0.37    518   +2/4      0.33


 4         +0.0073     +1.48    387   +2/4      0.25
 5         +0.0575     +9.35    311   +3/4      0.20


 6         +0.0824    +10.96    254   +3/4      0.17
 7         +0.0984    +11.02    218   +3/4      0.14


 8         +0.0812     +8.03    195   +3/4      0.12
 9         +0.0674     +5.93    171   +3/4      0.11


 10        +0.0761     +6.15    155   +3/4      0.10
 11        +0.1093     +8.28    145   +3/4      0.09


 12        +0.0898     +6.02    128   +3/4      0.08
 13        +0.0894     +5.42    117   +3/4      0.08


 14        +0.1008     +5.86    111   +3/4      0.07
